## Author: Gabriel Vance
## Class: DSC540 - Data Preparation
## Professor: Catherine Williams
## Date: July 5, 2025

# Purpose: This notebook performs various tasks labeled "Exercises" for Week 5/6 of the class. Data must be present within the folder that this notebook is run in.
# This week focuses on Data Cleaning and Transforming

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# This data had a unicode error, so we handle it with a try-except block.
try:
	candy_df = pd.read_csv('candyhierarchy2017.csv')
except UnicodeDecodeError:
	candy_df = pd.read_csv('candyhierarchy2017.csv', encoding='latin1')

In [3]:
# Display summary statistics
print(candy_df.describe(include='all'))

# Print the first few rows
print(candy_df.head())

# Print column names and their data types
print(candy_df.dtypes)

         Internal ID Q1: GOING OUT? Q2: GENDER Q3: AGE Q4: COUNTRY  \
count   2.460000e+03           2350       2419    2376        2396   
unique           NaN              2          4     106         128   
top              NaN             No       Male      40         USA   
freq             NaN           2038       1467      92         699   
mean    9.028016e+07            NaN        NaN     NaN         NaN   
std     7.435532e+03            NaN        NaN     NaN         NaN   
min     9.025877e+07            NaN        NaN     NaN         NaN   
25%     9.027520e+07            NaN        NaN     NaN         NaN   
50%     9.027780e+07            NaN        NaN     NaN         NaN   
75%     9.028242e+07            NaN        NaN     NaN         NaN   
max     9.031480e+07            NaN        NaN     NaN         NaN   

       Q5: STATE, PROVINCE, COUNTY, ETC Q6 | 100 Grand Bar  \
count                              2360               1713   
unique                             

# I chose these methods:

Chapter 7
* Transform data using mapping or a function
* Replace values

Chapter 8
* Create hierarchical index
* Pivot the data

Chapter 10
* Split/Apply/Combine
* Cross Tabs

Chapter 11
* Convert between string and date time
* Generate date range

Chapter 7 - Transform data using mapping or a function

In [ ]:
def clean_gender(gender):
    if pd.isnull(gender):
        return np.nan
    gender = str(gender).strip().lower()
    if gender in ['male', 'm']:
        return 'Male'
    elif gender in ['female', 'f']:
        return 'Female'
    else:
        return 'Other'

# Apply the function to the "Q2: GENDER" column
candy_df['Q2: GENDER_clean'] = candy_df['Q2: GENDER'].map(clean_gender)

# Display value counts of the cleaned gender column
print(candy_df['Q2: GENDER_clean'].value_counts(dropna=False))

Chapter 7 - Replace values

In [5]:
# Analyze the distribution of 'Q3: AGE'
age = pd.to_numeric(candy_df['Q3: AGE'], errors='coerce')
mean_age = age.mean()
median_age = age.median()
skewness = age.skew()

print(f"Mean: {mean_age:.2f}, Median: {median_age:.2f}, Skewness: {skewness:.2f}")

# This is a general function that I might want later. Skewness should determine whether or not we use mean or median.

# Decide replacement strategy
if abs(skewness) > 0.5:
    replacement = median_age
    print("Distribution is skewed. Replacing missing/invalid ages with median.")
else:
    replacement = mean_age
    print("Distribution is approximately normal. Replacing missing/invalid ages with mean.")

# Replace missing or invalid ages
candy_df['Q3: AGE_clean'] = age.fillna(replacement)

Mean: 42.61, Median: 41.00, Skewness: 28.20
Distribution is skewed. Replacing missing/invalid ages with median.


In [4]:
# I was going to transform these values, but the data is too messy.
# Instead, I will just print the unique values for context.
print(candy_df['Q4: COUNTRY'].unique())

[nan 'USA ' 'USA' 'us' 'usa' 'canada' 'Canada' 'Us' 'US' 'Murica'
 'United States' 'uk' 'United Kingdom' 'united states' 'Usa'
 'United States ' 'United staes' 'United States of America' 'UAE'
 'England' 'UK' 'canada ' 'Mexico' 'United states' 'u.s.a.' 'USAUSAUSA'
 'america' '35' 'france' 'United States of America ' 'U.S.A.' 'finland'
 'unhinged states' 'Canada ' 'united states of america' 'US of A'
 'Unites States' 'The United States' 'North Carolina ' 'Unied States'
 'Netherlands' 'germany' 'Europe' 'Earth' 'U S' 'u.s.' 'U.K. '
 'Costa Rica' 'The United States of America' 'unite states' 'U.S.' '46'
 'cascadia' 'Australia' 'insanity lately' 'Greece'
 'USA? Hard to tell anymore..' "'merica" 'usas' 'Pittsburgh' '45'
 'United State' '32' 'France' 'australia' 'A' 'Can' 'Canae' 'New York'
 'Trumpistan' 'Ireland' 'United Sates' 'Korea' 'California' 'Japan' 'USa'
 'South africa'
 'I pretend to be from Canada, but I am really from the United States.'
 'Usa ' 'Uk' 'Iceland' 'Germany' 'Canada`'

Chapter 8 - Create hierarchical index


In [ ]:
# Create a hierarchical index using 'Q4: COUNTRY' and 'Q2: GENDER_clean'
candy_df_hier = candy_df.set_index(['Q4: COUNTRY', 'Q2: GENDER'])


# A hierarchical index in pandas (also called a MultiIndex) allows you to have multiple levels 
# of row or column labels in a DataFrame. 
# Think of it as a way to organize your data in a tree-like structure, where each row or column can be
# identified by more than one key.




# Display the first few rows to show the new index
print(candy_df_hier.head())

                        Internal ID Q1: GOING OUT? Q3: AGE  \
Q4: COUNTRY Q2: GENDER                                       
NaN         NaN            90258773            NaN     NaN   
USA         Male           90272821             No      44   
USA         Male           90272829            NaN      49   
us          Male           90272840             No      40   
usa         Male           90272841             No      23   

                       Q5: STATE, PROVINCE, COUNTY, ETC Q6 | 100 Grand Bar  \
Q4: COUNTRY Q2: GENDER                                                       
NaN         NaN                                     NaN                NaN   
USA         Male                                     NM                MEH   
USA         Male                               Virginia                NaN   
us          Male                                     or                MEH   
usa         Male                               exton pa                JOY   

                  